# llm-finetune-serve — Colab driver

This notebook stays thin on purpose: clone, install, call scripts. All logic
lives in `src/`. If you find yourself writing real code here, it belongs in the
repo instead.

**Runtime → Change runtime type → GPU** before running anything.

In [ ]:
!nvidia-smi

## 1. Clone the repo

The repo is private, so Colab needs a token. In Colab: **key icon (Secrets) →
Add new secret**, name it `GITHUB_TOKEN`, paste a GitHub PAT with `repo` scope,
and enable notebook access. If the repo is later made public, the token is
ignored and the clone just works.

Re-running this cell pulls the latest commit rather than re-cloning.

In [ ]:
OWNER = "rushilpatra"
REPO = "llm-finetune-serve"
BRANCH = "main"

import os, subprocess

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = os.environ.get("GITHUB_TOKEN")

auth = f"{TOKEN}@" if TOKEN else ""
url = f"https://{auth}github.com/{OWNER}/{REPO}.git"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO], check=True)
%cd /content/llm-finetune-serve
!git pull --ff-only

## 2. Install

vLLM resolves its own torch build, so it goes first and the rest follows.
Colab will warn about a session restart — restart, then re-run the `%cd` cell
above and continue from here.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch, transformers

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("cuda        ", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    # T4 (Turing) has no bf16 support; scripts detect this at runtime.
    print("gpu         ", name)
    print("bf16        ", torch.cuda.is_bf16_supported())

## 3. Data smoke test

Prints split sizes, the 8-shot prefix length, and a sample prompt with the
round-trip answer-extraction check.

In [ ]:
!python -m src.data --split val --limit 2

## Next stages

Cells for training, evaluation, merging, and benchmarking get added here as
those scripts land. Each one is a single `!python -m src.<script> --config
configs/<run>.yaml` call.